# Day 6 — Virtual Environments, pip, requests & Public APIs (30 Oct)

*Rebuilt from the master Google Doc session (lesson text and every code example). Code cells are not executed: several need Postgres, a running server or files from the original starter zip.*

## Today at a glance

- Goal: write your first program that talks to another system over the network. You will give each project its own isolated Python (a virtual environment), install a third-party library with pip and record it, learn what an HTTP request and response really contain, and use requests the way production code does: timeouts, status checks, retries and pagination. Then you save the result as a CSV from a proper command-line tool.

- Deliverable: day06/cms_export.py, a CLI that pulls every blog post from a (local, fake) headless CMS API page by page, survives a server error, and writes a clean CSV. Plus day06/requirements.txt and day06/dsa.py. Stretch: day06/weather_csv.py against a real public API.

- Where this sits: Week 1, Phase 1 (Foundation), day 6 of 7. Everything in Phase 2 is an HTTP call: the Claude API, embedding APIs, vector databases and the customer systems you connect them to. Week 3's FastAPI is the other side of the same conversation, the server instead of the client.

| Block | Time | What you do |
|---|---|---|
| Theory | 1.5h | Concepts 1 to 5, running every example yourself |
| Hands-on | 2.5h | venv + requirements.txt, cms_export.py, all acceptance criteria |
| DSA | 0.5h | Chunk a list into pages, best time to buy and sell |
| Revision | 0.5h | Quiz, Remember cells, Day 4 and Day 5 open items |

Open items first: in the repo, Day 4's contact_store.py and main.py are still the starter TODOs, and there is no day05/ folder yet. If Day 5 exists on your machine, commit and push it first. Tomorrow is the weekly review, and it scores what the repo shows, so an unpushed day counts as not done.

## Recap (2 minutes, from memory)

- What does the recap code below print, and why is b.tags not affected by a.tags.append("sso")?

- Why is from_dict written as a @classmethod rather than a normal method? What does its first parameter receive?

- Your SupportTicket has the hint priority: Priority. Someone calls SupportTicket(..., priority="urgent") with a plain string and no __post_init__ check. Does Python stop them?

In [ ]:
# Recap: predict the output before running.
from dataclasses import dataclass, field


@dataclass
class Entry:
    uid: str
    tags: list[str] = field(default_factory=list)


a = Entry("blt1")
b = Entry("blt1")
a.tags.append("sso")
print(a == b, b.tags)

Answers are on the last page of today's session.

## Concept 1: Virtual environments

### Why it matters for an FDE

Every customer project you touch will need different libraries, often at different versions: one pins an old anthropic SDK, another needs the newest. If everything is installed into one shared Python, upgrading for project B quietly breaks project A. A virtual environment gives each project its own private set of packages, so "it works on my machine" becomes "it works from these exact versions", which is what you need when you hand code over to a client's team or deploy it into their servers.

### How it works

In Node, npm install puts packages in a node_modules/ folder next to package.json, so every project is isolated by default. Python is the opposite: pip install goes into the one global Python unless you say otherwise. A venv is how you say otherwise. python -m venv .venv creates a folder holding a small copy of the Python launcher and an empty site-packages folder. Activating it puts .venv/bin first on your terminal's PATH, so the words python and pip now mean the ones inside the folder.

```bash
# Create and use a virtual environment for day06 (macOS / Linux).
cd fde-journey/day06
python3 -m venv .venv  # make a private Python inside the .venv/ folder
source .venv/bin/activate  # switch THIS terminal to that Python
# Windows PowerShell instead:  .venv\Scripts\Activate.ps1
# your prompt now starts with (.venv), a reminder of where you are
which python  # Windows: where python
# -> /Users/rohit/fde-journey/day06/.venv/bin/python
python -m pip install requests  # lands in .venv only, nowhere else
deactivate  # back to the system Python (or just close the terminal)
```

Nothing magic is inside the folder. It is a launcher plus a place for packages, which is why you can delete .venv at any time and rebuild it in a minute from requirements.txt (Concept 2).

```bash
# What is inside .venv? (you never edit these files by hand)
.venv/
    bin/python  # a link to the real Python, so "python" means this one
    bin/pip  # a pip that installs into THIS environment
    lib/python3.13/site-packages/  # installed packages live here
        requests/  # like node_modules/axios/
    pyvenv.cfg  # notes which Python created the environment
```

Python can tell you which environment is running your code. Run this once inside the venv and once outside it.

In [ ]:
# Am I inside a virtual environment? Which Python is running me?
import sys

# sys.prefix is the folder of the Python running this code; base_prefix
# is the "real" install it came from. Different means: inside a venv.
print(sys.prefix != sys.base_prefix)  # -> True in .venv, False outside
print(sys.executable)  # -> .../day06/.venv/bin/python (path varies)

import requests  # works only if installed in THIS env

print(requests.__version__)  # -> 2.34.2 (or whatever you installed)

In VS Code, open the command palette, choose "Python: Select Interpreter" and pick the one under .venv. The editor, the terminal it opens and Jupyter notebooks then all use the same environment.

### Real-world FDE example

A bank gives you a locked-down Linux VM to deploy a document assistant. You cannot install anything system-wide, and their security team must approve every package. You create a venv in the project folder, install exactly the approved versions from a requirements.txt, and nothing outside that folder changes. Removing your work is one rm -rf .venv. In CMS terms, it is like a separate Contentstack stack per client: same platform, no content leaking between them.

### Common mistakes

- Opening a new terminal and forgetting to activate. The (.venv) prefix in the prompt is your check; ModuleNotFoundError: No module named 'requests' usually means the venv is not active.

- Typing pip install and getting a different Python's pip. Write python -m pip install ...: it always installs into the Python you are running.

- Committing .venv/ to Git. It holds thousands of files and only works on your OS. The repo's .gitignore already excludes it; commit requirements.txt instead.

- Fighting the "externally-managed-environment" error on macOS Homebrew or Linux by forcing a global install. That error is Python telling you to use a venv.

- Moving or renaming the project folder and expecting the venv to follow. It stores absolute paths; delete it and create a new one.

## Concept 2: pip and requirements.txt

### Why it matters for an FDE

The libraries you install are part of your deliverable. A customer's CI pipeline, a Docker image (Week 10) and a teammate's laptop all need to rebuild exactly the environment you tested with. That record is the requirements file, and reading one tells you quickly what a codebase depends on, which is the first thing you do when you join a customer project.

### How it works

pip downloads packages from PyPI (the Python Package Index, Python's npm registry) and installs them into the active environment. A package often needs other packages. requests pulls in four, and pip installs them automatically: these are transitive dependencies. pip freeze prints every installed package with its exact version, in the format pip install -r reads back.

```bash
# pip: install, record, and recreate dependencies (venv active).
python -m pip install "requests>=2.32,<3"  # any 2.x from 2.32 up
python -m pip list  # everything installed in this environment
python -m pip freeze > requirements.txt  # exact versions into a file
cat requirements.txt  # Windows: type requirements.txt
# -> certifi==2026.10.23
# -> charset-normalizer==3.5.2
# -> idna==3.20
# -> requests==2.34.2
# -> urllib3==2.8.0
# one package asked for, five installed: requests needs the other four
python -m pip install -r requirements.txt  # a teammate rebuilds yours
```

Version specifiers say how strict to be. In an application you deploy, pin exact versions (what pip freeze writes) so every build is identical. In a library others install, allow a range.

| Specifier | Means | Use it for |
|---|---|---|
| requests==2.34.2 | exactly this version | apps you deploy (reproducible) |
| requests>=2.32,<3 | 2.32 or newer, but not 3.x | libraries, or while you develop |
| requests~=2.34 | 2.34 or newer within 2.x | short form of the range above |
| requests | whatever is newest today | quick experiments only |

Mapped to what you already know from JavaScript:

| Node / npm | Python | Notes |
|---|---|---|
| node_modules/ | .venv/ | per-project packages, never committed |
| package.json | requirements.txt or pyproject.toml | what the project needs |
| package-lock.json | pinned requirements.txt or uv.lock | exact versions of everything |
| npm install | pip install -r requirements.txt | rebuild from the file |
| npm install axios | pip install requests | add one package |
| npmjs.com | pypi.org | the public registry |

You will also see uv, a newer tool written in Rust that does the same jobs much faster and is becoming the default in AI teams. Learn plain venv and pip first, because every server and Docker image has them. Switching to uv later is mostly a prefix change, and Week 2 moves the repo to pyproject.toml with uv.

```bash
# The same workflow with uv, a much faster installer (optional today).
uv venv  # creates .venv, like python -m venv .venv
source .venv/bin/activate
uv pip install requests  # same command shape as pip, seconds faster
uv pip freeze > requirements.txt
```

### Real-world FDE example

You inherit a customer's Python service with a two-line requirements.txt: requests and anthropic, no versions. It worked last month and fails today, because a fresh install pulled a new major version with a changed API. Pinning the versions from the last working environment fixes it in minutes, and you then upgrade one package at a time on purpose.

### Common mistakes

- Running pip freeze outside the venv and capturing every package on your laptop into the project's requirements.

- Editing requirements.txt by hand and forgetting to reinstall. The file is a record; pip install -r is what changes the environment.

- Installing a package because a tutorial said so, without checking it is the one you meant. Typo-squatted names exist on PyPI; read the name twice before installing.

- Leaving no version information at all for something you ship to a customer.

## Concept 3: HTTP, the conversation every API uses

### Why it matters for an FDE

An FDE's day is mostly connecting systems: a CMS to a vector database, a helpdesk to Claude, a CRM to a reporting sheet. Almost all of those connections are HTTP requests carrying JSON. When something breaks on a customer call, reading the status code, headers and body correctly is how you find out in two minutes whether it is your code, their API key or their server.

### How it works

HTTP is a request and a response. The request has a method (what you want to do), a URL (which thing), headers (extra facts such as the API key and the format you accept) and, for writes, a body. The response has a status code (what happened), headers and a body. You have used all of this through fetch() in React; fetch hid the parts you are about to look at.

| Method | Meaning | CMS example |
|---|---|---|
| GET | read, changes nothing | list blog posts |
| POST | create | create a new entry |
| PUT / PATCH | replace / partly update | edit an entry's title |
| DELETE | remove | delete an entry |

| Status | Family | What you do |
|---|---|---|
| 200, 201, 204 | 2xx success | read the body |
| 400, 422 | 4xx your request is wrong | fix the request; retrying will not help |
| 401, 403 | 4xx not allowed | check the key or permissions; never retry |
| 404 | 4xx not found | check the URL or the id |
| 429 | 4xx too many requests | wait (see Retry-After), then retry |
| 500, 502, 503, 504 | 5xx the server failed | retry with a growing wait, a few times |

To practise safely you need an API that behaves like a real customer's, including its bad days. day06/mock_cms.py is that: a small fake headless CMS shaped like Contentstack's Delivery API. It needs an api_key header, returns entries in pages and fails on purpose sometimes. Start it in its own terminal.

```bash
# Terminal 1: start the fake CMS and leave it running.
cd fde-journey/day06
python mock_cms.py
# -> mock CMS running on http://127.0.0.1:8006  (Ctrl+C to stop)
# each request your code makes shows up here as one log line:
# -> [mock_cms] GET /health -> 200
```

Now make a request in a second terminal or the notebook, and look at each part of the response.

In [ ]:
# Your first HTTP request: look at every part of the response.
import requests  # the standard HTTP client, like fetch() in JS

BASE = "http://127.0.0.1:8006"  # the mock CMS (other terminal)

r = requests.get(f"{BASE}/health", timeout=5)  # GET = "please give me"
print(r.status_code)  # -> 200  (2xx means success)
print(r.headers["Content-Type"])  # -> application/json; charset=utf-8
print(r.text)  # -> {"status": "ok"}   (the raw body, a str)
data = r.json()  # parse the JSON body, like await res.json() in JS
print(data["status"], type(data))  # -> ok <class 'dict'>
print(r.ok, r.elapsed.total_seconds() < 1)  # -> True True

The most important idea today: an error status is still a successful conversation. The server answered; it just said no.

In [ ]:
# The same API without the key: a 4xx is still a normal response.
import requests

BASE = "http://127.0.0.1:8006"
url = f"{BASE}/v3/content_types/blog_post/entries"

r = requests.get(url, timeout=5)  # no api_key header this time
print(r.status_code, r.ok)  # -> 401 False
print(r.json())  # -> {'error_message': 'Invalid API key', ...}
# GOTCHA: requests did NOT raise an error. A 401 or 500 is a valid HTTP
# reply, so your code keeps going unless YOU check the status.

r = requests.get(url, headers={"api_key": "demo-key-123"}, timeout=5)
print(r.status_code, len(r.json()["entries"]))  # -> 200 5

### Real-world FDE example

The real Contentstack Delivery API works the same way: a GET to /v3/content_types/blog_post/entries with api_key and access_token headers, and a 401 with an error_message field when the token is wrong. When a customer says "the integration is broken", the first question is the status code. A 401 is a rotated token on their side, a 503 is their outage, and a 200 with odd data is your parsing.

### Common mistakes

- Assuming a request "failed" only when an exception appears. With requests, a 404 or 500 raises nothing unless you ask it to.

- Calling .json() on an error page. Some servers return HTML on a 502, and .json() then raises a decoding error that hides the real problem. Check the status first.

- Putting API keys in the URL (?api_key=...). URLs end up in logs and browser history; send secrets in headers.

- Treating 4xx and 5xx the same. 4xx means change the request; 5xx means the same request may work in a moment.

## Concept 4: requests the production way

### Why it matters for an FDE

A tutorial call is requests.get(url).json(). A customer integration that runs every night over 40,000 records needs four more things: query parameters built safely, a timeout so a stuck server cannot freeze your job, a status check, and retries for the failures that fix themselves. The Anthropic SDK does exactly these things inside client.messages.create(); today you learn them by writing them.

### How it works

Pass query values as a dict with params=, and headers with headers=. requests builds and encodes the URL for you, the way URLSearchParams does in JavaScript.

In [ ]:
# Query parameters, headers and the final URL.
import requests

BASE = "http://127.0.0.1:8006"
url = f"{BASE}/v3/content_types/blog_post/entries"
headers = {"api_key": "demo-key-123"}  # a header, not part of the URL
# params: requests builds "?skip=0&limit=2&include_count=true" for you
params = {"skip": 0, "limit": 2, "include_count": "true"}

r = requests.get(url, params=params, headers=headers, timeout=5)
print(r.url)
# -> http://127.0.0.1:8006/v3/content_types/blog_post/entries?skip=0&...
body = r.json()
print(body["count"], [e["uid"] for e in body["entries"]])
# -> 8 ['blt001', 'blt002']
# FDE TIP: never glue user input into URLs by hand. params= encodes
# spaces, & and = safely, the way URLSearchParams does in JS.
print(requests.Request("GET", url, params={"q": "a&b c"}).prepare().url)
# -> http://127.0.0.1:8006/v3/content_types/blog_post/entries?q=a%26b+c

r.raise_for_status() turns a 4xx or 5xx into an exception, so the happy path stays clean. Network problems raise their own exceptions. All of them inherit from requests.RequestException, which is Day 5's inheritance doing real work.

In [ ]:
# raise_for_status() and the three failures every API client must handle.
import requests

BASE = "http://127.0.0.1:8006"
KEY = {"api_key": "demo-key-123"}


def get_json(path: str, headers: dict, timeout: float = 5) -> dict:
    """GET a path and return the JSON body, or raise a requests error."""
    r = requests.get(f"{BASE}{path}", headers=headers, timeout=timeout)
    r.raise_for_status()  # turns a 4xx/5xx reply into an HTTPError
    return r.json()


try:  # 1. the server answered, but with an error status
    get_json("/v3/content_types/blog_post/entries", headers={})
except requests.HTTPError as err:
    print("HTTP error:", err.response.status_code)  # -> HTTP error: 401
    print(err.response.json()["error_message"])  # -> Invalid API key

try:  # 2. the server is too slow (/slow takes 3 s, we wait 1 s)
    get_json("/slow", headers=KEY, timeout=1)
except requests.Timeout:
    print("timed out")  # -> timed out

try:  # 3. nothing is listening on that port at all
    requests.get("http://127.0.0.1:8999/health", timeout=2)
except requests.ConnectionError as err:
    print("cannot connect:", type(err).__name__)
    # -> cannot connect: ConnectionError

# REMEMBER: all three are children of requests.RequestException (Day 5's
# inheritance), so "except requests.RequestException" catches them all.
print(issubclass(requests.Timeout, requests.RequestException))  # -> True

GOTCHA: requests has no default timeout. Without timeout=, a server that accepts the connection and never answers keeps your program waiting forever. Every call you write gets one.

When you make several calls to the same API, use a Session. It keeps the connection open between calls (faster) and remembers headers, so the key is set in one place.

In [ ]:
# A Session reuses the connection and the headers for every call.
import requests

BASE = "http://127.0.0.1:8006"

# with: the session (and its open connections) closes automatically
with requests.Session() as session:
    # set once, sent on every request: like a configured axios instance
    session.headers.update({"api_key": "demo-key-123"})
    for skip in (0, 5):
        r = session.get(
            f"{BASE}/v3/content_types/blog_post/entries",
            params={"skip": skip, "limit": 5},
            timeout=5,  # GOTCHA: a Session has no default timeout either
        )
        r.raise_for_status()
        print(skip, [e["uid"] for e in r.json()["entries"]])
# -> 0 ['blt001', 'blt002', 'blt003', 'blt004', 'blt005']
# -> 5 ['blt006', 'blt007', 'blt008']

Some failures fix themselves: a 503 during a deploy, a 429 when you are going too fast, a dropped connection. Retry those a few times with a growing wait (exponential backoff), so you do not hammer a server that is already struggling. Never retry a 401 or 404: the same request will fail the same way.

In [ ]:
# Retry with backoff: try again only when trying again can help.
import time

import requests

BASE = "http://127.0.0.1:8006"
RETRY_STATUSES = {429, 500, 502, 503, 504}  # busy or broken right now


def get_with_retry(session: requests.Session, url: str, params: dict,
                   tries: int = 3, base_delay: float = 0.5) -> dict:
    """GET url; retry 429/5xx and connection errors, waiting longer."""
    for attempt in range(1, tries + 1):
        try:
            r = session.get(url, params=params, timeout=5)
        except requests.ConnectionError:
            if attempt == tries:
                raise  # out of tries: let the caller see the real error
            reason = "connection error"
        else:
            if r.status_code not in RETRY_STATUSES:
                r.raise_for_status()  # a 401 or 404 will not fix itself
                return r.json()
            if attempt == tries:
                r.raise_for_status()
            reason = f"HTTP {r.status_code}"
        delay = base_delay * 2 ** (attempt - 1)  # 0.5 s, 1 s, 2 s ...
        print(f"  {reason}, retry {attempt} of {tries - 1} in {delay}s")
        time.sleep(delay)
    raise RuntimeError("unreachable")  # loop always returns or raises


with requests.Session() as s:
    s.headers["api_key"] = "demo-key-123"
    url = f"{BASE}/v3/content_types/blog_post/entries"
    body = get_with_retry(s, url, {"skip": 3, "limit": 3})
    print([e["uid"] for e in body["entries"]])
# ->   HTTP 503, retry 1 of 2 in 0.5s
# -> ['blt004', 'blt005', 'blt006']

Finally, APIs return big collections in pages. You ask for skip=0&limit=5, then skip=5, and stop when a page is empty or you have the count the API reported. Contentstack, Drupal's JSON:API, HubSpot and Zendesk all do some version of this.

In [ ]:
# Pagination: keep asking for the next page until you have everything.
import requests

BASE = "http://127.0.0.1:8006"
url = f"{BASE}/v3/content_types/blog_post/entries"
PAGE_SIZE = 5

all_entries: list[dict] = []
with requests.Session() as s:
    s.headers["api_key"] = "demo-key-123"
    skip = 0
    while True:  # Day 2's while loop: we do not know the page count yet
        params = {"skip": skip, "limit": PAGE_SIZE,
                  "include_count": "true"}
        r = s.get(url, params=params, timeout=5)
        r.raise_for_status()
        body = r.json()
        page = body["entries"]
        all_entries.extend(page)  # extend, not append (Day 3)
        print(f"skip={skip}: got {len(page)}, total {len(all_entries)}")
        # stop on an empty page OR once we hold everything the API has
        if not page or len(all_entries) >= body["count"]:
            break
        skip += len(page)  # move forward by what we actually received
# -> skip=0: got 5, total 5
# -> skip=5: got 3, total 8
print(len(all_entries), all_entries[-1]["uid"])  # -> 8 blt008

The same tools work on any public API. This one returns Mumbai's forecast from Open-Meteo, which needs no key. It talks to the real internet, so your output will differ; it is today's stretch task.

In [ ]:
# A real public API: Mumbai's 3-day forecast from Open-Meteo (no key).
import requests

url = "https://api.open-meteo.com/v1/forecast"
params = {
    "latitude": 19.08,  # Mumbai
    "longitude": 72.88,
    "daily": "temperature_2m_max,temperature_2m_min",
    "timezone": "Asia/Kolkata",  # required when you ask for daily data
    "forecast_days": 3,
}
r = requests.get(url, params=params, timeout=10)
r.raise_for_status()
daily = r.json()["daily"]  # a dict of PARALLEL lists, one per field
# zip() walks several lists side by side, item i of each together
for day, hi, lo in zip(daily["time"], daily["temperature_2m_max"],
                       daily["temperature_2m_min"]):
    print(day, hi, lo)
# -> 2026-10-31 33.4 25.1   (live data: your dates and numbers differ)
# -> ...two more lines like it

### Real-world FDE example

A retailer wants every product description from their CMS indexed for a RAG assistant (Week 7). The export runs nightly over 120 pages. On page 40 the API returns 429 with Retry-After: 30. A naive script crashes and the index is half-built; yours waits 30 seconds, retries the same page, and finishes. On a later night the token has been rotated, and yours stops at once with "HTTP 401: Invalid API key" instead of retrying 120 times. Both behaviours are today's task.

### Common mistakes

- No timeout=, so one stuck request blocks the whole job.

- Retrying everything, including 401 and 404, or retrying forever with no limit and no wait.

- Pagination that never ends: forgetting to move skip forward, or trusting your limit instead of the number of entries that came back (servers cap page size; ours caps it at 5).

- all_entries.append(page) instead of extend(page), which gives you a list of lists (Day 3).

- Catching Exception around the request. You then also swallow your own bugs, such as a KeyError from parsing.

## Concept 5: CSV files, command-line tools and secrets

### Why it matters for an FDE

The person who asked for the data often lives in Excel or Google Sheets, so CSV is how a lot of FDE work is handed over. The script that produces it should run the same way for you, for a cron job and for the customer's ops team: a command with arguments, a clear exit code, and the API key read from the environment, never typed into the code.

### How it works

CSV looks simple until a value contains a comma or a quote. The csv module handles the escaping: a value with a comma is wrapped in quotes, and a quote inside it is doubled. csv.DictWriter writes one row per dict, with the columns in the order you list.

In [ ]:
# Writing CSV safely with csv.DictWriter.
import csv
from pathlib import Path

rows = [
    {"uid": "blt001", "title": "Getting started with SSO",
     "tags": "sso|auth"},
    # a comma and quotes INSIDE a value: the csv module escapes them
    {"uid": "blt002",
     "title": 'Pricing FAQ: "Starter" vs "Pro", explained',
     "tags": "pricing"},
]
path = Path("posts_demo.csv")
# newline="": the csv module controls line endings (else blank lines on
# Windows). encoding="utf-8": accents survive. Both are Day 4 habits.
with open(path, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["uid", "title", "tags"])
    writer.writeheader()  # the first line: uid,title,tags
    writer.writerows(rows)  # one line per dict, fieldnames order

print(path.read_text(encoding="utf-8"))
# -> uid,title,tags
# -> blt001,Getting started with SSO,sso|auth
# -> blt002,"Pricing FAQ: ""Starter"" vs ""Pro"", explained",pricing

Reading it back with csv.DictReader gives you dicts again, but CSV has no types: every value is a string.

In [ ]:
# Reading CSV back: every value comes back as a str.
import csv

with open("posts_demo.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):  # each row: a dict keyed by header
        print(row["uid"], row["title"][:20], row["tags"].split("|"))
# -> blt001 Getting started with ['sso', 'auth']
# -> blt002 Pricing FAQ: "Starte ['pricing']
# GOTCHA: CSV has no types. A word_count of 820 comes back as "820", and
# True comes back as "True". Convert with int() when you read it.

FDE TIP: Excel on Windows often shows accented names like Renée as garbage when it opens a plain UTF-8 CSV. If the file is going to Excel users, write it with encoding="utf-8-sig", which adds a marker Excel recognises. Python and Google Sheets read both.

argparse turns the words typed after the script name into an object with typed attributes, and prints a usage message and exits with code 2 when the input is wrong. Secrets come from environment variables: os.environ is Python's process.env.

In [ ]:
# argparse turns command-line words into a typed object.
import argparse
import os

parser = argparse.ArgumentParser(description="Export CMS entries.")
parser.add_argument("content_type")  # positional: required, no dashes
parser.add_argument("--out", default="export.csv")  # optional flag
parser.add_argument("--page-size", type=int, default=5)  # gives an int

# parse_args() normally reads sys.argv (what you typed in the terminal);
# passing a list lets us test it without a terminal.
args = parser.parse_args(["blog_post", "--page-size", "3"])
print(args.content_type, args.out, args.page_size)
# -> blog_post export.csv 3
print(type(args.page_size))  # -> <class 'int'>
# GOTCHA: "--page-size" becomes args.page_size (dash -> underscore)

# Secrets come from the environment, never from the code or the CLI args.
# os.environ is a dict of env variables, like process.env in Node.
key = os.environ.get("CMS_API_KEY")  # None if it was never set
print(key is None)  # -> True (until you run: export CMS_API_KEY=...)

Two more conventions make a CLI scriptable. Print normal output to stdout and errors to stderr with print(..., file=sys.stderr), and end with sys.exit(code): 0 means success, anything else means failure, which is what cron, CI and shell scripts check.

### Real-world FDE example

Before migrating a customer from Drupal to Contentstack, the content team wants a spreadsheet of every article with its author, tags and word count, to decide what to archive. A one-off script becomes python cms_export.py article --out audit.csv, which they can re-run themselves next week with their own key. That is exactly the shape of today's task.

### Common mistakes

- Forgetting newline="" when opening a CSV file for writing, which puts a blank line between rows on Windows.

- Building CSV lines by hand with ",".join(values), which breaks on the first title with a comma.

- Writing a Python list straight into a cell. You get the text "['sso', 'auth']"; join it into one string such as "sso|auth".

- Hard-coding the API key and pushing it to GitHub. Treat any key that reached a public repo as leaked: rotate it.

- bool("False") after reading a CSV. Any non-empty string is True; compare the text instead.

## Check your understanding (predict first, then run)

Start mock_cms.py first. Write down your prediction for each numbered part, then run it in the notebook or a scratch file.

In [ ]:
# Predict each output BEFORE you run this cell.
import csv
import io

import requests

BASE = "http://127.0.0.1:8006"
URL = f"{BASE}/v3/content_types/blog_post/entries"
KEY = {"api_key": "demo-key-123"}

# 1
r = requests.get(URL, timeout=5)
print(r.ok, r.status_code)

# 2
r = requests.get(URL, params={"skip": 0, "limit": 99}, headers=KEY,
                 timeout=5)
print(len(r.json()["entries"]))

# 3
buf = io.StringIO()  # a fake file in memory, handy for quick tests
csv.writer(buf).writerow(["a,b", 'say "hi"', 3])
print(buf.getvalue().strip())

# 4
row = next(csv.DictReader(io.StringIO("uid,published\nblt003,False\n")))
print(row["published"], bool(row["published"]))

# 5
try:
    e0 = requests.get(URL, headers=KEY, timeout=5).json()["entries"][0]
    e0["slug"]
except requests.RequestException:
    print("network problem")
except KeyError as err:
    print("missing field:", err)

## Hands-on task (about 2.5 hours): day06/cms_export.py

A customer's content team wants every blog post from their headless CMS in a spreadsheet before a migration. Their API is paginated, needs a key and has a flaky server. Build a command-line tool that fetches everything, retries only when that can help, stops with one clear message when it cannot, and writes a clean CSV.

```bash
# day06/ layout for today's task
day06/
    mock_cms.py  # the fake customer CMS (given, do not edit)
    cms_export.py  # YOUR CLI: fetch every entry, write a CSV
    requirements.txt  # YOU create it with pip freeze
    weather_csv.py  # stretch: the same idea against a real public API
    dsa.py  # today's two DSA problems
    day06_notes.ipynb  # runnable examples from this session
    .venv/  # your virtual environment (git ignores it, never commit)
```

Step 1, environment (about 20 min): inside day06/, create and activate .venv, install requests, and write requirements.txt with pip freeze. Select the venv in VS Code.

Step 2, the tool: the starter file gives you the constants, the CmsApiError exception and the CSV column order. You write:

- get_api_key(): read CMS_API_KEY from the environment; raise CmsApiError with the exact message shown below if it is missing.

- fetch_page(session, base_url, content_type, skip, limit, tries=3): one page with include_count=true and a 5-second timeout. Return the JSON on 200. Retry 429/5xx (wait Retry-After seconds if the header is there, else 0.5 s, 1 s, 2 s) and connection errors or timeouts. Any other status raises CmsApiError("HTTP <code>: <error_message>") at once.

- fetch_all_entries(session, base_url, content_type, page_size): the pagination loop, printing page N: K entries per page.

- entry_to_row(entry): one flat dict for the CSV. Missing author becomes "", tags are joined with "|", published becomes "yes"/"no", the title is stripped.

- write_csv(rows, path): create the parent folder, write header and rows, return the row count.

- main(argv=None): argparse with a positional content_type, --out (default <content_type>.csv), --page-size (int, default 3) and --base-url. Return 0, 1 or 2 as shown; errors go to stderr.

```bash
# What a correct run looks like (restart mock_cms.py first)
$ export CMS_API_KEY=demo-key-123  # PowerShell: $env:CMS_API_KEY="..."
$ python cms_export.py blog_post --out output/posts.csv
page 1: 3 entries
  HTTP 503 on skip=3, retrying in 1.0s (attempt 1 of 3)
page 2: 3 entries
page 3: 2 entries
wrote 8 rows to output/posts.csv
```

```bash
# The failure cases: one clear line on stderr, no traceback
$ CMS_API_KEY=wrong python cms_export.py blog_post
error: HTTP 401: Invalid API key  # exit code 1, and NO retry
$ python cms_export.py product
error: HTTP 422: Content type 'product' was not found  # exit code 1
$ python cms_export.py blog_post --base-url http://127.0.0.1:8999
  connection failed on skip=0, retrying in 0.5s (attempt 1 of 3)
  connection failed on skip=0, retrying in 1.0s (attempt 2 of 3)
error: cannot reach http://127.0.0.1:8999 after 3 attempts  # exit 1
$ unset CMS_API_KEY; python cms_export.py blog_post
error: CMS_API_KEY is not set. Run: export CMS_API_KEY=demo-key-123
# exit code 2 (wrong setup, not a server problem)
```

```bash
# output/posts.csv, first lines (long rows cut with ... here)
uid,title,author,tags,word_count,published,updated_at
blt001,Getting started with SSO,Priya Nair,sso|auth,820,yes,...
blt002,"Pricing FAQ: ""Starter"" vs ""Pro"", explained",Rahul Mehta,...
blt003,Draft: Q4 roadmap,,,150,no,2026-10-05T08:15:00Z
```

Acceptance criteria:

- With a freshly restarted mock_cms.py, the run above prints exactly those five lines and exits with code 0 (check with echo $?).

- output/posts.csv has the header line shown, 8 data rows in uid order, the blt002 title correctly quoted, an empty author and tags for blt003, and Renée's name readable.

- The four failure cases print exactly the lines shown (on stderr), exit with the codes shown and never print a traceback. The 401 and 422 cases make exactly one request each: watch the server log.

- Every request has a timeout; there is no bare except: and no except Exception.

- requirements.txt exists in day06/, was made by pip freeze inside the venv, and pins requests. .venv/ is not committed.

- No API key appears anywhere in cms_export.py.

- python -c "import cms_export" run inside day06/ prints nothing.

- Stretch: weather_csv.py writes Mumbai's 7-day forecast to output/forecast.csv with columns date, max_c, min_c.

- Committed and pushed with a meaningful message.

Hints (no solutions, try first):

- Build bottom-up. Write entry_to_row and write_csv first and test them on one hand-written dict; they need no network.

- Concept 4's get_with_retry is the shape of fetch_page. What is new: r.headers.get("Retry-After"), converting connection errors into CmsApiError on the last attempt, and reading error_message from the error body.

- Concept 4's pagination loop is the shape of fetch_all_entries. Count pages with a separate counter; do not compute it from skip.

- Use raise CmsApiError(...) from err so the original cause is kept if you ever need it while debugging.

- In main, catch only CmsApiError, print error: <message> to stderr and return the code. sys.exit(main()) is already written for you at the bottom.

- The mock fails every other request to the skip=3 page. Restart it before checking criterion 1, or your first run may not show the retry line.

## DSA practice (30 min)

- Chunk a list into pages. chunk(items: list, size: int) -> list[list] splits a list into pages of at most size items, the same thing an API does on the server. chunk([1, 2, 3, 4, 5, 6, 7], 3) gives [[1, 2, 3], [4, 5, 6], [7]]; chunk([], 3) gives []; a size below 1 raises ValueError. Hint: range(0, len(items), size) gives the start of each page, and slicing past the end is safe.

- Best time to buy and sell. max_profit(prices: list[int]) -> int: buy on one day, sell on a later day, return the best profit or 0. [7, 1, 5, 3, 6, 4] gives 5 (buy at 1, sell at 6); [7, 6, 4, 3, 1] gives 0. One loop, O(n). Hint: as you walk the list, remember the cheapest price so far and ask what selling today would earn.

## Quiz (3 quick questions)

- What problem does a virtual environment solve, and which file lets a teammate rebuild yours? What are the npm equivalents?

- A nightly export gets a 429 on page 40, and on another night a 401 on page 1. What should your client do in each case, and why are they different?

- Why must every requests call have a timeout=, and what happens without one?

## What you achieved today

You can isolate a project in a virtual environment, install and pin packages with pip and a requirements file (and know where uv fits), read any HTTP exchange through its method, URL, headers, status and body, and call an API like production code does: safe parameters, timeouts, status checks, retries with backoff for the right errors only, and pagination. You can also write correct CSV, build a CLI with argparse and keep secrets in environment variables. That is the client side of every integration in this plan.

Tomorrow (Day 7): weekly review. Scores per syllabus area from what the repo shows, a mock interview, your LinkedIn post announcing the FDE journey, and the plan for Week 2.

## When you're done

Paste into the Forward Deployment Engineer project chat as "Day 6 review": your recap answers, your predictions next to the real outputs, cms_export.py, your requirements.txt, the terminal output of the main run and the four failure cases, both DSA solutions and the quiz answers. Include anything finished from Day 4 and Day 5. Tracebacks are welcome; send them with what you tried.

## Recap answers

*Your answers:*
